In [ ]:
# From shenmeyemeifashengguo/STATE-ToxiCN, api.ipynb, the one cell that runs the
# benchmark's test split through an OpenAI compatible endpoint. The repository's
# model, gemini-2.0-flash-exp, is not in the catalog, so gemini-2.0-flash is
# substituted. The prompt is Chinese in the original and is kept that way, along
# with the dictionary literal whose repeated keys leave only the user message.

# Please install OpenAI SDK first: `pip3 install openai`

import json
from openai import OpenAI

try:
    client = OpenAI(api_key="your_key", base_url="your_url")

    with open('/data/test.json', 'r', encoding='utf-8') as file:
        sentences = json.load(file)

    def generate_prompt(sentence):
        return {
            'role': 'system', 'content': '你是一个内容审查专家，请你分析我的句子并且从中提取出一个或者多个四元组',
            'role': 'user', 'content': f'''请从下面的文本抽取一个或多个四元组，每一个四元组输出格式为评论对象|对象观点|仇恨群体|是否仇恨，评论对象可以为"NULL", 对象观点尽量简洁,仇恨群体只包括(LGBTQ、Region、Sexism、Racism、others、non-hate)，同一四元组可能涉及多个仇恨群体，是否仇恨标签为(hate、non-hate),多个四元组之间用[SEP]分隔,最后一个四元组后面加[END]。
            仇恨样本输入："谁喜欢小黑啊，黑不拉几的看一眼都嫌脏，也就某些连汽车排气口都不放过的男人会喜欢黑妞"
            仇恨样本输出："小黑|黑不拉几的看一眼都嫌脏|Racism|hate [SEP] 某些连汽车排气口都不放过的男人|喜欢黑妞|Sexism, Racism|hate [END]"
            非仇恨样本输入："由同性伴侣抚养长大的孩子，在学校表现比异性伴侣抚养的孩子更好，并且比异性伴侣的孩子高中毕业率高出4.8%。"
            非仇恨样本输出："由同性伴侣抚养长大的孩子|在学校表现比异性伴侣抚养的孩子更好|non-hate|non-hate [END]"
            提取出句子中包含的所有四元组:"{sentence}"'''
        }

    output = []

    def save_results():
        with open('your_test_file.json', 'w', encoding='utf-8') as file:
            json.dump(output, file, ensure_ascii=False, indent=4)

    for sentence in sentences:
        try:
            completion = client.chat.completions.create(
                model="gemini-2.0-flash",
                messages=[
                    generate_prompt(sentence)
                ]
            )

            result = completion.choices[0].message.content.strip()

            output.append({
                'sentence': sentence,
                'response': result
            })

            save_results()

        except Exception as e:
            print(f"Error：{e}")
            output.append({
                'sentence': sentence,
                'response': f"Error: {e}"
            })

    print("Processing completed, results have been saved to the output.json file.")

except Exception as e:
    print(f"Initialization error：{e}")
